<a href="https://colab.research.google.com/github/ajitamishra308/flyrank-ml-internship-ajita/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents one query-document pair for a specific reporting period in the FlyRank search warehouse.

## Time Window

For this notebook, I use a mid-panel month (March 2026) to avoid using the final month, which is reserved as the outcome window. This helps prevent data leakage while building features.

In [ ]:
print("Unit of Analysis: One query-document pair")
print("Time Window: March 2026 (mid-panel month)")

Unit of Analysis: One query-document pair
Time Window: March 2026 (mid-panel month)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
- Impressions
- Clicks
- CTR
- Average Position

## Label
Ranking relevance score.

## Context
- Client Hash ID
- Content Hash ID
- Report Date

## Excluded
Final month (June 2026) is excluded because it can leak future information into training.


In [ ]:
features = [
    "Impressions",
    "Clicks",
    "CTR",
    "Average Position"
]

label = "Ranking Relevance Score"

context = [
    "Client Hash ID",
    "Content Hash ID",
    "Report Date"
]

excluded = "June 2026 (avoids leakage)"

print("Features:", features)
print("Label:", label)
print("Context:", context)
print("Excluded:", excluded)

Features: ['Impressions', 'Clicks', 'CTR', 'Average Position']
Label: Ranking Relevance Score
Context: ['Client Hash ID', 'Content Hash ID', 'Report Date']
Excluded: June 2026 (avoids leakage)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

# Verification Queries

The following queries verify the data contract:

1. Grain verification
2. Row count and date span
3. Availability check

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
summary = con.sql(f"""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS first_date,
MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE year(report_date)=2026
AND month(report_date)=3
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.